## Extract and process Somatic data from cBioPortal

Search for the term 'USER' to locate codes lines where user must provide input

In [1]:
# 1) import modules
import os
import re
import json
import pandas as pd
import scipy
import time
import requests
import hashlib
import csv
import random
from collections import defaultdict
import numpy as np    
import statsmodels.api as sm   
import seaborn as sns
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

In [ ]:
# 2) filter and process somatic data 
#  (A) cBioPortal 

# see R script: "Extract_cBioPortal.R" for extracting somatic data from cBioPortal gene-wise 

# stored extracted data files in the respective individial gene folders

# stored clinical data from all samples in a seaparte clinical datafile in parentpath (information to annotate variants with tumor mutation burden, cancer type detailed/oncotree code for cancer type the variant was observed in)

# reading in from there and processing below:

start_time=time.asctime(time.localtime())
print(start_time)
#setting path to input and output files
#i) 
parentpath="USER INSERT PARENT PATH TO FOLDER WHERE YOU WILL CREATE SUB-FOLDERS PER GENE OF INTEREST AND OUTPUT EXTRACTED DATA"

os.chdir(parentpath)

#read file with folder names and MANE Select transcript identifiers (or MANE Plus Clinical in the case of SMARCA4 for this study) and Gene ID
#ii) USER INPUT FILE NAME 
TranscriptID= pd.read_csv("USER INPUT FILE NAME CONTAINING GENE SYMBOL+NCBI GENE ID+ MANE SELECT TRANSCRIPT ID (Supplemental Table)", sep="\t")

# read file with chromosome identifiers on GRCh37 to construct HGVSg for cBioPortal data
GRCh37_chr_id= pd.read_csv("USER INSERT PATH/GRCh37_Assembly_Chromosome_identifiers_8-7-23.txt", sep="\t")

# read file with clinical data (R script extraction)
cbio_sampleinfo=pd.read_csv("USER INSERT PATH/cBio_all_clinical_fromRStudio_6-14-23.txt", sep="\t")
# filter down to rows that contain sampleid + Tumor Mutation Burden (TMB) information:
cbio_sampleinfo_tmb=cbio_sampleinfo[cbio_sampleinfo["clinicalAttributeId"].str.contains("TMB")==True]
# convert TMB values column to numeric to be able to filter by value:
cbio_sampleinfo_tmb["value"]=pd.to_numeric(cbio_sampleinfo_tmb["value"])

# filter to TMB greater than 10 - make list of samples to be exluded from analysis:
Hypermutators=cbio_sampleinfo_tmb[cbio_sampleinfo_tmb["value"]>10]

os.chdir("USER INSERT PATH")

#initialize dataframe to compile variant counts at each filter/processing step:
allTSGdf_cbio_counts=pd.DataFrame()

# Loop through each folder name aka each gene in the list to read the respective extracted variant file 
for index, row in TranscriptID.iterrows():
    
    folder_name=TranscriptID.loc[index]["Gene Symbol"]
    #first load MANE Select/transcript identifier to a variable
    transcriptname=TranscriptID.loc[index]["MANE Select Transcript ID"]
    #if folder name exists, list all files in that folder then change directory to that folder
    if os.path.exists(folder_name) and os.path.isdir(folder_name):
        filelist=os.listdir(folder_name)
        os.chdir(folder_name)
        
        #cBio raw data file name starts with 'cBio_' so find that filename from list of filenames in folder and load into pandas df (short for dataframe)
        infile_cbio=f"cBio_{folder_name}_fromRStudio_"
        for file in filelist:
            if file.startswith(infile_cbio):
                cbio_rawdata = pd.read_csv(file, sep="\t")
                
                
        # count of variants:
        One_cbioall=len(cbio_rawdata)
        cbio_rawdata_unique=cbio_rawdata.drop_duplicates(subset=['chr', 'startPosition', 'endPosition', 'referenceAllele', 'variantAllele'])
        One_cbioall_unique=len(cbio_rawdata_unique)
        
        #cbio processing:
        #removing uncalled and germline variants
        cbio_rawdata_tos=cbio_rawdata[cbio_rawdata['mutationStatus'].str.contains("UNCALLED|Germline|GERMLINE|germline")==True]
        cbio_rawdata_s = cbio_rawdata.drop(cbio_rawdata_tos.index)
        
        #s=somatic
        Two_somatic=len(cbio_rawdata_s)
        cbio_rawdata_s_unique=cbio_rawdata_s.drop_duplicates(subset=['chr', 'startPosition', 'endPosition', 'referenceAllele', 'variantAllele'])
        Two_somatic_unique=len(cbio_rawdata_s_unique)
        
        #u=unique meaning removing duplicate variants in patients with multiple samples (by patient id)
        cbio_rawdata_s_u=cbio_rawdata_s.drop_duplicates(subset=['patientId', 'chr', 'startPosition', 'endPosition', 'referenceAllele', 'variantAllele'])
        Three_uniqueperpatient=len(cbio_rawdata_s_u)
        cbio_rawdata_s_u_unique=cbio_rawdata_s_u.drop_duplicates(subset=['chr', 'startPosition', 'endPosition', 'referenceAllele', 'variantAllele'])
        Three_uniqueperpatient_unique=len(cbio_rawdata_s_u_unique)

        # filter out the variants that have 0 reads of the tumor allele (likely invalid entries)
        cbio_rawdata_s_u_0tumorAltCount=cbio_rawdata_s_u[cbio_rawdata_s_u["tumorAltCount"]==0]
        cbio_rawdata_s_u_no0tumorAltCount=cbio_rawdata_s_u.drop(cbio_rawdata_s_u_0tumorAltCount.index)
        print(len(cbio_rawdata_s_u_no0tumorAltCount))
        Four_no0tumorAltCount=len(cbio_rawdata_s_u_no0tumorAltCount)
        cbio_rawdata_s_u=cbio_rawdata_s_u_no0tumorAltCount
        cbio_rawdata_s_u_unique=cbio_rawdata_s_u.drop_duplicates(subset=['chr', 'startPosition', 'endPosition', 'referenceAllele', 'variantAllele'])
        Four_no0tumorAltCount_unique=len(cbio_rawdata_s_u_unique)

        #QC step error checking for genomic coordinates
        #inserting column for QC result
        cbio_rawdata_s_u.insert(1,"QCgenomiccoordinates","NaN")
        #looping over every row in dataframe
        for index, row in cbio_rawdata_s_u.iterrows():
            #assigning row and column indexers to respective variables to be able to call a specific position in df
            rowindex=cbio_rawdata_s_u.index.get_loc(index)
            colindex=cbio_rawdata_s_u.columns.get_loc('QCgenomiccoordinates')
            #length of ref allele minus 1 should match difference between stop and start (except for insertions where diff should be 1 and ref="-")
            if (int(row['endPosition'])-int(row['startPosition']))==(len(row["referenceAllele"])-1):
                cbio_rawdata_s_u.iloc[rowindex,colindex]="Pass"
            else:
                if ((int(row['endPosition'])-int(row['startPosition']))==1)&(row["referenceAllele"]=="-"): 
                    cbio_rawdata_s_u.iloc[rowindex,colindex]="Pass"
                #if criteria not satisfied, variant Fails QC
                else:
                    cbio_rawdata_s_u.iloc[rowindex,colindex]="Fail"
        #filter out failed variants at QC step
        cbio_rawdata_s_u_p=cbio_rawdata_s_u[cbio_rawdata_s_u['QCgenomiccoordinates'].str.contains("Pass")==True]
        ##print QC fail variants to new file to see why they failed (f = fail)
        
        os.chdir(parentpath)
        os.chdir("USER INSERT PATH")

        
        cbio_rawdata_s_u_f=cbio_rawdata_s_u[cbio_rawdata_s_u['QCgenomiccoordinates'].str.contains("Fail")==True]
        #cbio_rawdata_s_u_f.to_csv("cbio_rawdata_s_u_f.txt", sep="\t")
        Five_passgenomicQC=len(cbio_rawdata_s_u_p)
        Five_failgenomicQC=len(cbio_rawdata_s_u_f)
        cbio_rawdata_s_u_p_unique=cbio_rawdata_s_u_p.drop_duplicates(subset=['chr', 'startPosition', 'endPosition', 'referenceAllele', 'variantAllele'])
        Five_passgenomicQC_unique=len(cbio_rawdata_s_u_p_unique)

        
        ##using OncoKB API to annotate variants with somatic classification aka whether it is cancer associated or not
        ##making json file for OncoKB API
        json_for_oncoKB=[]
        
        with open('json_for_oncokb_python.json','w') as outfile:
            for index, row in cbio_rawdata_s_u_p.iterrows():
                #adding id to json file to identify the input variant- this identifier will be printed in output and used to match the output with input info for each variant
                json_for_oncoKB.append({"genomicLocation":f"{row['chr']},{row['startPosition']},{row['endPosition']},{row['referenceAllele']},{row['variantAllele']}","id":f"{index}","referenceGenome":"GRCh37"})
            json.dump(json_for_oncoKB, outfile)
        ##querying oncoKB api
        url_oncoKBAPI="https://www.oncokb.org/api/v1/annotate/mutations/byGenomicChange"
        #ENTERyourKEYhere
        headersnewkey = {'Authorization': 'USER INSERT API KEY', 'Accept' : 'application/json', 'Content-Type' : 'application/json'}
        
        
        r_oncoKBAPI = requests.post(url_oncoKBAPI,data=open('json_for_oncokb_python.json', 'rb'), headers=headersnewkey)
        OncoKBAPI_output=pd.read_json(r_oncoKBAPI.text)
        #making new column with the input id in oncokb output file so can join back to cbio-raw-data file using index in the correct order
        OncoKBAPI_output.insert(0,"index_id","NaN")
        for index, row in OncoKBAPI_output.iterrows():
            rowindex=OncoKBAPI_output.index.get_loc(index)
            colindex=OncoKBAPI_output.columns.get_loc('index_id')
            query_identifier=row["query"]
            OncoKBAPI_output.iloc[rowindex,colindex]=query_identifier.get("id")
        #converting id from string to integer
        OncoKBAPI_output['index_id']=pd.to_numeric(OncoKBAPI_output['index_id'])
        #setting row index of the oncokb api output df to id (which is technically the row index of the input file) so now can use this common row index to match and join the 2 files
        OncoKBAPI_output_resetindex=OncoKBAPI_output.set_index('index_id')

        #joining df from OncoKB API with processed cbiorawdata file:
        cbio_rawdata_s_u_p_oncokb=cbio_rawdata_s_u_p.join(OncoKBAPI_output_resetindex)

        Six_OncoKBannotated=len(cbio_rawdata_s_u_p_oncokb)
        
        #olo=oncogenic or likely oncogenic annotation filter
        cbio_rawdata_s_u_p_olo=cbio_rawdata_s_u_p_oncokb[cbio_rawdata_s_u_p_oncokb['oncogenic'].str.contains("Oncogenic|Likely Oncogenic")==True]
        Seven_OLO=len(cbio_rawdata_s_u_p_olo)
        print(Seven_OLO)
        
        # filter out variants in hypermutated samples
        Hypermutators_to_filter=Hypermutators.set_index('uniqueSampleKey').index
        cbio_nohypermutator=cbio_rawdata_s_u_p_olo[~cbio_rawdata_s_u_p_olo["uniqueSampleKey"].isin(Hypermutators_to_filter)]
        cbio_rawdata_s_u_p_olo=cbio_nohypermutator
        Eight_TMBgt10=len(cbio_rawdata_s_u_p_olo)
        
        #variables for chr number of gene from gene data and then finding that chr number in GRCh37 identifier df
        loop=0
        for index, row in cbio_rawdata_s_u_p_olo.iterrows():
            if loop==0:
                chrnumber=int(cbio_rawdata_s_u_p_olo.loc[index]["chr"])
                loop=1    
        chrindex=chrnumber-1
        chridentifier=str(GRCh37_chr_id.loc[chrindex]["Identifier"])

        #constructing hgvs:
        ##script to convert genomic coordinates to HGVSg: 
        cbio_rawdata_s_u_p_olo.insert(0,"HGVSg","NaN")
        for index, row in cbio_rawdata_s_u_p_olo.iterrows():
            rowindex=cbio_rawdata_s_u_p_olo.index.get_loc(index)
            colindex=cbio_rawdata_s_u_p_olo.columns.get_loc('HGVSg')
            start=cbio_rawdata_s_u_p_olo.loc[index]["startPosition"]
            stop=cbio_rawdata_s_u_p_olo.loc[index]["endPosition"]
            ref=cbio_rawdata_s_u_p_olo.loc[index]["referenceAllele"]
            alt=cbio_rawdata_s_u_p_olo.loc[index]["variantAllele"]
            #if SNV HGVSg format is chridentifier:g.start ref>alt, and so on for different types of variants below
            if ((start==stop)&(len(ref)==len(alt))&(ref!="-")&(alt!="-")):
                SNV= (f"{chridentifier}:g.{start}{ref}>{alt}")
                cbio_rawdata_s_u_p_olo.iloc[rowindex,colindex]=SNV
        
            else: 
                if ((start==stop)&(ref!="-")&(alt=="-")):
                    SNVdel= (f"{chridentifier}:g.{start}del")
                    cbio_rawdata_s_u_p_olo.iloc[rowindex,colindex]=SNVdel
               
                else:
                    if ((start!=stop)&(ref!="-")&(alt=="-")):
                        deletion= (f"{chridentifier}:g.{start}_{stop}del")
                        cbio_rawdata_s_u_p_olo.iloc[rowindex,colindex]=deletion

                    else:
                        if (ref=="-"):
                            ins= (f"{chridentifier}:g.{start}_{stop}ins{alt}")
                            cbio_rawdata_s_u_p_olo.iloc[rowindex,colindex]=ins
                
                        else:
                            if ((start!=stop)&(ref!="-")&(alt!="-")):
                                delins= (f"{chridentifier}:g.{start}_{stop}delins{alt}")
                                cbio_rawdata_s_u_p_olo.iloc[rowindex,colindex]=delins
                
                            else:
                                if ((start==stop)&(ref!="-")&(alt!="-")&(len(ref)!=len(alt))):
                                    singledelins= (f"{chridentifier}:g.{start}delins{alt}")
                                    cbio_rawdata_s_u_p_olo.iloc[rowindex,colindex]=singledelins
                            
        ###ClinGen Allele Registry API (CAR API) lift over to MANE Select transcript:
        #drop duplicate HGVSg to query CAR API faster and will append back the annotation to file with # variants as many times as has been seen in database/pop
        cbio_rawdata_s_u_p_olo_unique=cbio_rawdata_s_u_p_olo.drop_duplicates(subset=['HGVSg'])
        
        #save newly created HGVSg column to .txt file to input in bulk to CAR API
        cbio_rawdata_s_u_p_olo_unique["HGVSg"].to_csv("cbio_rawdata_s_u_p_olo_unique_HGVSg.txt", sep="\t", index=False)
        #cbio TERT all promoter region- no MANE Select HGVSc so lifting over to GRCh38 HGVSg only for TERT
        
        #MANE and not MANE.nucleotide... because need MANE Status in output to select MANE Select and not Mane plus clinical- PTCH1, TP63, CDKN2A
        url_CARAPI="https://reg.clinicalgenome.org/alleles?file=hgvs&fields=none+transcriptAlleles.MANE"
        res_CARAPI=requests.post(url_CARAPI,data=open('cbio_rawdata_s_u_p_olo_unique_HGVSg.txt').read())
        CARAPI_output=pd.read_json(res_CARAPI.text)


        #join CARAPI output back to cbio dataframe first by joining to input hgvsg since expected to be in same order, and then using hgvsg column to join back to processed raw data file:
        #read HGVSg file to df
        cbio_rawdata_s_u_p_olo_HGVSg = pd.read_csv('cbio_rawdata_s_u_p_olo_unique_HGVSg.txt', sep="\t", header=None)
        #join CAR API output to HGVSg input file by numerical index
        cbio_rawdata_s_u_p_olo_HGVSg_CARAPI=cbio_rawdata_s_u_p_olo_HGVSg.join(CARAPI_output)
        #drop 1st row with error since 'HGVSg' header in first row
        cbio_rawdata_s_u_p_olo_HGVSg_CARAPI=cbio_rawdata_s_u_p_olo_HGVSg_CARAPI.drop(0)
        #rename HGVSg column from input file joined to output file
        cbio_rawdata_s_u_p_olo_HGVSg_CARAPI_rename_unique=cbio_rawdata_s_u_p_olo_HGVSg_CARAPI.rename(columns={0:"HGVSg"})
        #reset index to HGVSg column to join with that processed rawdata file
        cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex = cbio_rawdata_s_u_p_olo_HGVSg_CARAPI_rename_unique.set_index('HGVSg')
        
        #make column with HGVSc from CAR API output transcript alleles column
        
        #add empty column for HGVSc
        cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.insert(1,"HGVSc","NaN")
        #fill empty column row by row (self explanatory code below)
    
        
        if folder_name!="SMARCA4": #(since do not want MANE select for SMARCA4, want MANE Plus Clinical instead)
            for index, row in cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.iterrows():
                rowindex=cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.index.get_loc(index)
                colindex=cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.columns.get_loc('HGVSc')
                hgvsc=[]
                if str(cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.loc[index]['transcriptAlleles'])!='nan':
                    for dictionary in cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.loc[index]['transcriptAlleles']:
                        if "MANE" in dictionary:
                            if "MANE Select" in dictionary["MANE"]["maneStatus"]:
                                hgvsc.append(dictionary["MANE"]["nucleotide"]["RefSeq"]["hgvs"])
                            else:
                                #2 variants in CDKN2A that are upstream of MANE Select, HGVS from CAR API only on Mane Plus clinical
                                if folder_name=="CDKN2A":
                                    if "MANE Plus Clinical" in dictionary["MANE"]["maneStatus"]:
                                        hgvsc.append(dictionary["MANE"]["nucleotide"]["RefSeq"]["hgvs"])
                        else:
        #if variant is not registered in CAR, MANE status is not indicated, but output does contain hgvsc on the MANE Select transcript, so select those by matching hgvs output with the transcript id
                            if "hgvs" in dictionary:
                                for value in dictionary["hgvs"]:
                                    if value.startswith(f"{transcriptname}:"):
                                        hgvsc.append(value)

                    cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.iloc[rowindex,colindex] = hgvsc[0]
        
                else:
                    cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.iloc[rowindex,colindex] = "error"
        else:#meaning for SMARCA4 alone
            for index, row in cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.iterrows():
                rowindex=cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.index.get_loc(index)
                colindex=cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.columns.get_loc('HGVSc')
                hgvsc=[]
                if str(cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.loc[index]['transcriptAlleles'])!='nan':
                    for dictionary in cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.loc[index]['transcriptAlleles']:
                        if "MANE" in dictionary:
                            if "MANE Plus Clinical" in dictionary["MANE"]["maneStatus"]:
                                hgvsc.append(dictionary["MANE"]["nucleotide"]["RefSeq"]["hgvs"])
                    #if variant is not registered in CAR, MANE status is not indicated, but output does contain hgvsc on the MANE Select transcript, so select those by matching hgvs output with the transcript id
                        else:
                            if "hgvs" in dictionary:
                                for value in dictionary["hgvs"]:
                                    if value.startswith(f"{transcriptname}:"):
                                        hgvsc.append(value)

                    cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.iloc[rowindex,colindex] = hgvsc[0]
        
                else:
                    cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.iloc[rowindex,colindex] = "error"
                    
        #drop rows which had 'error' output from CAR API (QC if there is an error and, df output at this step to check what and why)
        cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors=cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex[cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex['HGVSc'].str.contains("error")==True]
        cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_no_errors = cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_resetindex.drop(cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.index)
        cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.to_csv("cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_errors.txt", sep="\t")
        
        #set HGVSg as index for processed raw data file so that can join with CAR API output processed file by HGVSg column
        cbio_rawdata_s_u_p_olo_resetindex = cbio_rawdata_s_u_p_olo.set_index('HGVSg')
        cbio_rawdata_s_u_p_olo_CARAPI = cbio_rawdata_s_u_p_olo_resetindex.join(cbio_rawdata_s_u_p_olo_HGVSg_CARAPIunique_no_errors)
        
        Nine_CARAPI=len(cbio_rawdata_s_u_p_olo_CARAPI)
        
        #save this file to .txt file as a checkpoint here. Until here is the most time-consuming part of code, so if have to make changes downstream, can start with this checkpoint file instead of redoing the OncoKB and CAR API codes:
        cbio_rawdata_s_u_p_olo_CARAPI.to_csv("cbio_rawdata_s_u_p_olo_CARAPI.txt", sep="\t")
        
        #adding filter to drop duplicate HGVSc in same patient (when the HGVSg might be different but ultimately results in the same protein change or HGVSc-sometimes these are listed multiple times)
        cbio_processed_rawdata=cbio_rawdata_s_u_p_olo_CARAPI.drop_duplicates(subset=['patientId', 'HGVSc'])
        
        Ten_finaldupdrop=len(cbio_processed_rawdata)
        
        #print HGVSc column with only unique HGVSc to .txt file for annotations (CA ID and VEP together with germline processed data)
        cbio_processed_rawdata_toannotate=cbio_processed_rawdata.drop_duplicates(subset=['HGVSc'])
        Eleven_uniqueHGVSc=len(cbio_processed_rawdata_toannotate)
        print(Eleven_uniqueHGVSc)
        cbio_processed_rawdata_toannotate["HGVSc"].to_csv("cbio_processed_rawdata_unique_HGVSctoannotate.txt", sep="\t", index=False, header=None)
        
        #print processed file for downstream analysis
        cbio_processed_rawdata.to_csv("cbio_processed_rawdata.txt", sep="\t")

        
        outputdf=pd.DataFrame({'Gene':[folder_name],'One_cbioall': [One_cbioall], 'Two_somatic':[Two_somatic], 'Three_uniqueperpatient':[Three_uniqueperpatient], 'Four_no0tumorAltCount':[Four_no0tumorAltCount], 'Five_passgenomicQC':[Five_passgenomicQC],'Five_failgenomicQC':[Five_failgenomicQC], 'Six_OncoKBannotated':[Six_OncoKBannotated],'Seven_OLO':[Seven_OLO], 'Eight_TMBgt10':[Eight_TMBgt10], 'Nine_CARAPI':[Nine_CARAPI], 'Ten_finaldupdrop':[Ten_finaldupdrop], 'Eleven_uniqueHGVSc':[Eleven_uniqueHGVSc]})
        allTSGdfcounts=pd.concat([allTSGdfcounts,outputdf])
       

        #checkpoint for where the code is while running- print gene name (directory)
        print("Current directory:", os.getcwd())
        #go back to parent directory after completing gene and start over with next gene
        #os.chdir(parentpath)
    else:
        print(f"Folder '{folder_name}' not found.")

end_time=time.asctime(time.localtime())
print(end_time)